# NVD API Queries

Query the NVD REST API 2.0 directly for targeted CVE lookups, keyword searches,
and filtered queries. Results are returned as a `VulnDataset`.

**Rate limits:** 5 requests/30s without an API key, 50/30s with one.  
Set `NVD_API_KEY` env var or pass `api_key=` to bypass the public limit.

## Single CVE Lookup

In [ ]:
from vulnflow.extractors.nvd_api import NVDAPIExtractor

# The xz backdoor (CVE-2024-3094)
ds = NVDAPIExtractor(cve_id="CVE-2024-3094").extract()
df = ds.to_pandas(include_nested=True)
print(df.iloc[0].to_string())

## Keyword Search

In [ ]:
ds = NVDAPIExtractor(keyword="log4shell", no_rejected=True).extract()
ds.transform()
df = ds.to_pandas()
print(f"{len(df):,} results for 'log4shell'\n")
df[["cve_id", "cvss_base_score", "description"]].head(10)

## Severity + Date Filter

In [ ]:
import matplotlib.pyplot as plt

ds = NVDAPIExtractor(
    cvss_v3_severity="CRITICAL",
    pub_start_date="2025-01-01T00:00:00+00:00",
    pub_end_date="2025-01-31T00:00:00+00:00",
    no_rejected=True,
).extract()
ds.transform()
df = ds.to_pandas()
print(f"{len(df):,} CRITICAL CVEs in Jan 2025\n")

# Attack vector distribution
vectors = df["cvss_attack_vector"].value_counts()
VECTOR_NAMES = {"NETWORK": "Network", "LOCAL": "Local", "ADJACENT_NETWORK": "Adjacent", "PHYSICAL": "Physical"}
labels = [VECTOR_NAMES.get(v, v) for v in vectors.index]

fig, ax = plt.subplots(figsize=(8, 6))
ax.bar(labels, vectors.values, color="#1f77b4", edgecolor="white")
for i, (lbl, val) in enumerate(zip(labels, vectors.values)):
    ax.text(i, val + 1, f"{val:,}", ha="center", fontsize=12, fontweight="bold")
ax.set_title("Critical CVEs by Attack Vector (Jan 2025)", fontsize=16)
ax.set_ylabel("Count", fontsize=12)
ax.grid(True, linestyle="--", alpha=0.5, axis="y")
plt.tight_layout()
plt.show()

## Date Range Auto-Chunking

The NVD API limits date ranges to 120 days. `NVDAPIExtractor` automatically splits
wider ranges into chunks, paginates each, and deduplicates results.

In [ ]:
# 6-month range — auto-chunked into ~2 requests
ds = NVDAPIExtractor(
    pub_start_date="2025-01-01T00:00:00+00:00",
    pub_end_date="2025-06-30T00:00:00+00:00",
    no_rejected=True,
).extract()
print(f"{len(ds):,} CVEs from Jan-Jun 2025 (auto-chunked)")

## KEV-Only Entries

In [ ]:
ds = NVDAPIExtractor(
    has_kev=True,
    pub_start_date="2024-01-01T00:00:00+00:00",
    pub_end_date="2024-12-31T00:00:00+00:00",
).extract()
ds.transform()
df = ds.to_pandas()
print(f"{len(df):,} CVEs with KEV entries published in 2024\n")
df[["cve_id", "cvss_base_score", "description"]].head(10)

## Pipeline Integration

Use `NVDAPIExtractor` inside a `VulnPipeline` for programmatic workflows.

In [ ]:
from vulnflow import VulnPipeline

result = VulnPipeline(
    extractors=[NVDAPIExtractor(keyword="log4j", no_rejected=True)],
).run()

print(f"Records: {len(result.dataset):,}")
print(f"Duration: {result.duration:.1f}s")
print(f"Stages: {result.successful_stages}")
print(f"Errors: {len(result.errors)}")

## Enrich API Results

API results come without enrichments. Call `transform()` to add EPSS, KEV, CWE, etc.

In [ ]:
# Reuse the pipeline result from above
ds = result.dataset
print(f"Before transform: {len(ds.columns)} columns")

ds.transform()
print(f"After transform: {len(ds.columns)} columns")

df = ds.to_pandas()
df[["cve_id", "cvss_base_score", "epss_probability", "is_kev", "cwe_id"]].head(10)

## CVE Change History

The `NVDHistoryEnricher` fetches change history for CVEs from the NVD API.

In [ ]:
import json
from vulnflow.transforms.enrichers.nvd_history import NVDHistoryEnricher

# Fetch a single high-profile CVE and its history
ds = NVDAPIExtractor(cve_id="CVE-2024-3094").extract()
ds.transform(transforms=[NVDHistoryEnricher(mode="per_cve")])

df = ds.to_pandas(include_nested=True)
changes = df["nvd_changes"].iloc[0]
if isinstance(changes, str):
    changes = json.loads(changes)

print(f"CVE-2024-3094: {len(changes)} change events\n")
for c in changes[:5]:
    print(f"  {c.get('created', 'N/A')} — {c.get('action', 'N/A')}: {c.get('type', 'N/A')}")